# Dijital İkiz İnsan — Türkçe Duygu Polaritesi Pilot Deneyi

Bu bağımsız notebook, Türkçe metinlerde **olumlu / nötr / olumsuz** sınıflandırması için ilk ölçülebilir baselinedır. Mevcut VAD ve Big Five regresyon defterini değiştirmez.

**Bilimsel sınır:** Bu veri kümesinin etiketleri duygu polaritesidir; sürekli Valence-Arousal-Dominance (VAD) ölçümü veya Big Five kişilik etiketi değildir. Bu deneyin sonucu tek başına dijital ikiz başarısını ya da %75 kişilik/duygu doğruluğunu kanıtlamaz.

Veri: [winvoker/turkish-sentiment-analysis-dataset](https://huggingface.co/datasets/winvoker/turkish-sentiment-analysis-dataset), CC BY-SA 4.0. Veri kartı kaynağın yaklaşık 489 bin örnek içerdiğini, olumlu/olumsuz/nötr sınıfları kullandığını ve farklı kaynaklardan derlendiğini bildirir. Atıf ve lisans koşullarına uy; proje raporunda veri kaynağını belirt. Veri kartı: https://huggingface.co/datasets/winvoker/turkish-sentiment-analysis-dataset/blob/main/README.md

Notebook, verinin tamamında uzun eğitim yapmak yerine sabit tohumla küçük bir pilot alt kümesi kullanır. Gerçek sonuç `metrics.json` içinde saklanır; sonuç çalıştırılmadan başarı oranı raporlanmaz.

## 1. Ortamı hazırla

Kaggle ayarlarında Internet açık olsun. Erişim sağlanmazsa doğrulamayı atlatma; hesabın resmi ayarlarını kontrol et. GPU zorunlu değil ama eğitim CPU'da çok yavaş olabilir.

In [ ]:
import sys, subprocess, random, json, math, re, unicodedata, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51,<5', 'accelerate>=1.8', 'safetensors>=0.5', 'scikit-learn>=1.6'], check=True)

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Cihaz:', 'GPU' if torch.cuda.is_available() else 'CPU')
print('Torch:', torch.__version__)
print('Transformers ortamı hazır.')

## 2. Veriyi indir ve kontrollü biçimde temizle

Sürüm sabitlenmiş `train.csv` ve `test.csv` dosyaları indirilir. Aynı normalize edilmiş metin hem eğitimde hem testte bulunursa eğitim kopyası çıkarılır. Etiketleri birbiriyle çelişen aynı metinler analizden çıkarılır. Bu veri kümesinde katılımcı kimlikleri yoktur; dolayısıyla bu, **örnek düzeyinde** bir testtir ve görülmemiş kişiler için genelleme kanıtı değildir.

In [ ]:
DATA_REVISION = 'ef87c7a2a7199a8b721c99425933316d35f9af2f'
BASE_URL = f'https://huggingface.co/datasets/winvoker/turkish-sentiment-analysis-dataset/resolve/{DATA_REVISION}'
WORK = Path('/kaggle/working/digital_twin_sentiment')
WORK.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = WORK / 'train.csv'
TEST_PATH = WORK / 'test.csv'
for filename, destination in [('train.csv', TRAIN_PATH), ('test.csv', TEST_PATH)]:
    if not destination.exists():
        url = f'{BASE_URL}/{filename}?download=true'
        print('İndiriliyor:', filename)
        urllib.request.urlretrieve(url, destination)
    print(filename, 'MB:', round(destination.stat().st_size / (1024 * 1024), 2))

def canonical_label(value):
    text = unicodedata.normalize('NFKD', str(value)).casefold()
    text = ''.join(char for char in text if not unicodedata.combining(char)).replace('ı', 'i').strip()
    aliases = {
        'positive': 'positive', 'pozitif': 'positive', 'olumlu': 'positive',
        'negative': 'negative', 'negatif': 'negative', 'olumsuz': 'negative',
        'neutral': 'neutral', 'notr': 'neutral', 'neutral sentiment': 'neutral',
    }
    return aliases.get(text)

def clean_frame(frame, name):
    required = {'text', 'label'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{name}: beklenen text/label kolonları yok: {sorted(missing)}; bulunan: {list(frame.columns)}')
    result = frame[['text', 'label']].copy()
    result['text'] = result['text'].fillna('').astype(str).str.replace(r'[\x00-\x1F\x7F]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
    result['label'] = result['label'].map(canonical_label)
    result = result[(result['text'] != '') & result['label'].notna()].copy()
    result['_key'] = result['text'].str.casefold()
    conflicts = result.groupby('_key')['label'].nunique()
    conflict_keys = set(conflicts[conflicts > 1].index)
    conflict_count = len(conflict_keys)
    result = result[~result['_key'].isin(conflict_keys)].drop_duplicates('_key', keep='first')
    print(f'{name}: temiz örnek={len(result):,}; çelişkili metin anahtarı={conflict_count:,}; sınıflar={result.label.value_counts().to_dict()}')
    return result.reset_index(drop=True)

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_df = clean_frame(train_raw, 'train')
test_df = clean_frame(test_raw, 'test')
del train_raw, test_raw

test_keys = set(test_df['_key'])
before_overlap = len(train_df)
train_df = train_df[~train_df['_key'].isin(test_keys)].reset_index(drop=True)
removed_overlap = before_overlap - len(train_df)
if removed_overlap:
    print(f'Test ile aynı normalize metne sahip eğitim satırı çıkarıldı: {removed_overlap:,}')
if train_df['label'].nunique() != 3 or test_df['label'].nunique() != 3:
    raise ValueError('Her iki bölümde de üç sınıfın bulunması gerekir; veri sürümünü ve kolonları kontrol et.')
if len(train_df) < 30 or len(test_df) < 30:
    raise ValueError('Eğitim veya test verisi beklenenden küçük.')
print('Çapraz bölüm metin çakışması:', len(set(train_df['_key']) & set(test_df['_key'])))

## 3. Pilot örneklemi ve model

En çok 15.000 eğitim ve 3.000 doğrulama/test örneği alınır. Sınıf başına örnekleme ve sabit tohum kullanılır. Daha büyük çalışma gerekirse bu sınırlar artırılabilir; fakat sonuçlar artık farklı bir deney konfigürasyonu olur.

In [ ]:
def stratified_limit(frame, limit, seed=SEED):
    if limit is None or limit <= 0 or len(frame) <= limit:
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    classes = sorted(frame['label'].unique())
    per_class = max(1, limit // len(classes))
    parts = []
    for offset, label in enumerate(classes):
        group = frame[frame['label'] == label]
        parts.append(group.sample(n=min(len(group), per_class), random_state=seed + offset))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

train_pool, validation_pool = train_test_split(
    train_df, test_size=0.1, random_state=SEED, stratify=train_df['label']
)
train_pilot = stratified_limit(train_pool, 15000)
validation_pilot = stratified_limit(validation_pool, 3000, SEED + 1)
test_pilot = stratified_limit(test_df, 3000, SEED + 2)
print('Pilot örnek sayıları:', {'train': len(train_pilot), 'validation': len(validation_pilot), 'test': len(test_pilot)})
print('Train etiket dağılımı:', train_pilot['label'].value_counts().to_dict())
print('Validation etiket dağılımı:', validation_pilot['label'].value_counts().to_dict())
print('Test etiket dağılımı:', test_pilot['label'].value_counts().to_dict())

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}
MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
MAX_LENGTH = 160
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextClassificationDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_LENGTH):
        self.texts = frame['text'].astype(str).tolist()
        self.labels = [LABEL2ID[label] for label in frame['label'].tolist()]
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, index):
        item = self.tokenizer(self.texts[index], truncation=True, max_length=self.max_length)
        item['labels'] = self.labels[index]
        return item

train_dataset = TextClassificationDataset(train_pilot, tokenizer)
validation_dataset = TextClassificationDataset(validation_pilot, tokenizer)
test_dataset = TextClassificationDataset(test_pilot, tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenizer ve veri kümeleri hazır.')

## 4. Eğit ve test et

Model seçimi doğrulama kümesindeki macro-F1 ile yapılır. En son bağımsız test alt kümesi bir kez değerlendirilir. Sınıf bazındaki precision/recall/F1 ve confusion matrix rapora yazılır.

In [ ]:
def compute_metrics(prediction):
    logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
    predicted = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(prediction.label_ids, predicted)),
        'f1_macro': float(f1_score(prediction.label_ids, predicted, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(prediction.label_ids, predicted, average='weighted', zero_division=0)),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, label2id=LABEL2ID, id2label=ID2LABEL
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
output_dir = Path('/kaggle/working/artifacts/sentiment')
output_dir.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(output_dir / 'checkpoints'),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to='none',
    seed=SEED,
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)
trainer.train()
test_output = trainer.predict(test_dataset, metric_key_prefix='test')
test_predictions = np.argmax(test_output.predictions[0] if isinstance(test_output.predictions, tuple) else test_output.predictions, axis=-1)
test_truth = test_output.label_ids
majority_label = train_pilot['label'].value_counts().idxmax()
majority_predictions = np.full(len(test_truth), LABEL2ID[majority_label], dtype=int)
majority_baseline = {
    'majority_class': majority_label,
    'accuracy': float(accuracy_score(test_truth, majority_predictions)),
    'f1_macro': float(f1_score(test_truth, majority_predictions, average='macro', zero_division=0)),
}
test_metrics = {
    'majority_class_baseline': majority_baseline,
    'accuracy': float(accuracy_score(test_truth, test_predictions)),
    'f1_macro': float(f1_score(test_truth, test_predictions, average='macro', zero_division=0)),
    'f1_weighted': float(f1_score(test_truth, test_predictions, average='weighted', zero_division=0)),
    'per_class': classification_report(test_truth, test_predictions, labels=[0, 1, 2], target_names=[ID2LABEL[i] for i in range(3)], output_dict=True, zero_division=0),
    'confusion_matrix_labels_negative_neutral_positive': confusion_matrix(test_truth, test_predictions, labels=[0, 1, 2]).tolist(),
}
trainer.save_model(str(output_dir / 'model'))
tokenizer.save_pretrained(str(output_dir / 'model'))
report = {
    'task': 'Turkish three-class sentiment polarity classification',
    'model': MODEL_NAME,
    'dataset': 'winvoker/turkish-sentiment-analysis-dataset',
    'dataset_revision': DATA_REVISION,
    'dataset_license_declared_by_hub': 'CC BY-SA 4.0',
    'dataset_reference': 'https://huggingface.co/datasets/winvoker/turkish-sentiment-analysis-dataset',
    'seed': SEED,
    'max_length': MAX_LENGTH,
    'epochs': 2,
    'batch_size': 8,
    'train_rows': len(train_pilot),
    'validation_rows': len(validation_pilot),
    'test_rows': len(test_pilot),
    'training_rows_removed_due_to_test_overlap': int(removed_overlap),
    'test_metrics': test_metrics,
    'scientific_limitations': [
        'Bu deney yalnızca olumlu/nötr/olumsuz polarite sınıflandırır; VAD boyutlarının tamamını ölçmez.',
        'Bu deney Big Five kişilik tahmini veya kişi düzeyinde dijital ikiz doğrulaması değildir.',
        'Kaynak veri kümesinde katılımcı kimliği bulunmadığı için değerlendirme örnek düzeyindedir.',
        '3.000 örnekli test alt kümesindeki metrikler bu deney konfigürasyonuna aittir; genellenebilir başarı iddiası olarak yorumlanmamalıdır.'
    ]
}
metrics_path = output_dir / 'metrics.json'
metrics_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'test_metrics': test_metrics, 'metrics_path': str(metrics_path)}, ensure_ascii=False, indent=2))

## 5. Sonuç paketini indir

ZIP dosyası model ağırlıklarını ve `metrics.json` raporunu içerir. Bu pilotun gerçek sonuçları yalnızca notebook çalıştırılıp dosya üretildikten sonra rapora eklenmelidir.

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/dijital-ikiz-sentiment-pilot', 'zip', str(output_dir))
print('İndirilecek paket:', archive)
print('Gerçek deney metrikleri:')
print(metrics_path.read_text(encoding='utf-8'))